# Retriever training

## Runtime Bootstrap

In [ ]:
# Runtime bootstrap: local / Google Colab / Kaggle
import os
import sys
import subprocess

from pathlib import Path
import importlib.util

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle").exists()
RUNTIME = "colab" if IN_COLAB else "kaggle" if IN_KAGGLE else "local"

# Help CUDA allocator reduce fragmentation (must be set before importing torch).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Keep this False for normal training speed. Turn on only when debugging CUDA asserts.
DEBUG_CUDA_ASSERT = False
if DEBUG_CUDA_ASSERT:
    os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / "utils").exists() and (p / "data").exists():
            return p
    return start


def _find_repo_root_in_kaggle_input() -> Path | None:
    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.exists():
        return None

    # Search a few levels for a folder containing both utils/ and data/.
    for depth_glob in ["*", "*/*", "*/*/*"]:
        for p in kaggle_input.glob(depth_glob):
            if p.is_dir() and (p / "utils").exists() and (p / "data").exists():
                return p.resolve()
    return None


def _ensure_on_syspath(path: Path):
    path_str = str(path.resolve())
    if path_str not in sys.path:
        sys.path.insert(0, path_str)


def _install_missing_packages(pkg_to_module):
    missing = [pkg for pkg, mod in pkg_to_module if importlib.util.find_spec(mod) is None]
    if not missing:
        return
    print(f"Installing missing packages: {missing}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


candidate_starts = [Path.cwd()]
if IN_COLAB:
    candidate_starts = [
        Path("/content/Legal-question-answer-v1"),
        Path("/content/drive/MyDrive/Colab Notebooks/NLP/Legal-question-answer-v1"),
        Path.cwd(),
    ]
elif IN_KAGGLE:
    candidate_starts = [Path.cwd(), Path("/kaggle/working/Legal-question-answer-v1")]
    kaggle_input_root = _find_repo_root_in_kaggle_input()
    if kaggle_input_root is not None:
        candidate_starts.append(kaggle_input_root)

REPO_ROOT = None
for c in candidate_starts:
    if c.exists():
        root = _find_repo_root(c)
        if (root / "utils").exists():
            REPO_ROOT = root
            break
if REPO_ROOT is None:
    REPO_ROOT = _find_repo_root(Path.cwd())

_ensure_on_syspath(REPO_ROOT)

if IN_COLAB or IN_KAGGLE:
    _install_missing_packages(
        [
            ("transformers", "transformers"),
            ("scikit-learn", "sklearn"),
            ("tqdm", "tqdm"),
            ("peft", "peft"),
        ]
    )

if IN_COLAB:
    print("Running in Google Colab")
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print(f"Drive mount skipped: {e}")
elif IN_KAGGLE:
    print("Running in Kaggle Notebook")
    print("Use /kaggle/input for read-only datasets and /kaggle/working for outputs.")
else:
    print("Running in local environment")

print(f"Runtime: {RUNTIME}")
print(f"Repo root: {REPO_ROOT}")
print(f"PYTORCH_CUDA_ALLOC_CONF={os.environ.get('PYTORCH_CUDA_ALLOC_CONF')}")

if DEBUG_CUDA_ASSERT:
    print("CUDA_LAUNCH_BLOCKING=1 enabled for debugging.")
    print("After fixing errors, set DEBUG_CUDA_ASSERT=False for faster training.")
else:
    print("CUDA_LAUNCH_BLOCKING disabled for faster training.")

## Load Training Data

In [ ]:
import json
from pathlib import Path
import importlib.util

# REPO_ROOT is already prepared in the bootstrap cell; ensure it is importable.
repo_root = REPO_ROOT if isinstance(globals().get("REPO_ROOT"), Path) else Path.cwd()
repo_root_str = str(repo_root.resolve())
if repo_root_str not in sys.path:
    sys.path.insert(0, repo_root_str)

ROOT = REPO_ROOT if isinstance(globals().get("REPO_ROOT"), Path) else Path.cwd()

def _load_get_jsonl_data_func():
    # Preferred import when repo layout is available as a Python package path.
    try:
        from utils.get_jsonl_data import get_jsonl_data as fn
        return fn
    except Exception:
        pass

    # Fallback to direct file load for Kaggle dataset mounts.
    candidate_files = [
        ROOT / "utils" / "get_jsonl_data.py",
        Path("/kaggle/input/datasets/trinhthaison/legal-question-answer-v1/utils/get_jsonl_data.py"),
    ]
    for py_file in candidate_files:
        if py_file.exists():
            spec = importlib.util.spec_from_file_location("get_jsonl_data_dynamic", str(py_file))
            if spec is not None and spec.loader is not None:
                module = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(module)
                return module.get_jsonl_data

    raise ModuleNotFoundError(
        "Cannot load get_jsonl_data. Ensure utils/get_jsonl_data.py exists in repo or Kaggle input dataset."
    )


def resolve_training_data_path() -> str:
    candidates = [
        Path("/kaggle/input/datasets/trinhthaison/training-data-v1-legal-question-answer-v1/training_data_v1.jsonl"),
        Path("/content/drive/MyDrive/Colab Notebooks/NLP/Legal-question-answer-v1/data/training_data_v1.jsonl"),
        ROOT / "data" / "training_data_v1.jsonl",
    ]

    for p in candidates:
        if p.exists():
            print(f"training_data path: {str(p)}")
            return str(p)

    if IN_KAGGLE:
        # Common Kaggle pattern: /kaggle/input/<dataset-name>/...
        found = list(Path("/kaggle/input").glob("**/training_data_v1.jsonl"))
        if found:
            return str(found[0])

    raise FileNotFoundError(
        "Cannot find training_data_v1.jsonl. Place it under data/ in repo, "
        "or attach a Kaggle dataset containing training_data_v1.jsonl."
    )


get_jsonl_data = _load_get_jsonl_data_func()
training_data_jsonl_path = resolve_training_data_path()
print(f"Loading training data from: {training_data_jsonl_path}")
training_data = get_jsonl_data(training_data_jsonl_path)
training_data[0]

## Training Configuration and Imports

In [ ]:
import random
import copy
import math
from dataclasses import dataclass, field
from typing import Dict, List

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from peft import LoraConfig, TaskType, get_peft_model

# Fail fast if torch is partially initialized in notebook kernel state.
if not hasattr(torch, "fx"):
    torch_file = getattr(torch, "__file__", "<unknown>")
    raise RuntimeError(
        "Your current kernel has a broken/partial torch import (torch.fx missing). "
        "Restart the kernel, then run from Cell 1 again. "
        f"Loaded torch from: {torch_file}"
    )

# ----------------------------
# 1) Reproducibility settings
# ----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Torch file: {getattr(torch, '__file__', '<unknown>')}")
print(f"Torch version: {getattr(torch, '__version__', '<unknown>')}")

if torch.cuda.is_available():
    # Slight speed/memory win on Ampere+ without changing model outputs materially.
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark = True

@dataclass
class TrainConfig:
    model_name: str = "vinai/phobert-base"
    # Use shorter lengths for faster iteration. Increase later if recall drops.
    max_q_len: int = 96
    max_c_len: int = 256
    batch_size: int = 16
    grad_accum_steps: int = 2
    max_epochs: int = 7
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    temperature: float = 0.05
    grad_clip: float = 1.0
    warmup_ratio: float = 0.1
    early_stopping_patience: int = 2
    num_negatives: int = 5
    dataloader_num_workers: int = 4
    dataloader_persistent_workers: bool = True
    # Multi-negative training needs cache generated with matching num_negatives.
    use_pretokenized_cache: bool = False
    pretokenized_cache_path: str = "/kaggle/input/datasets/trinhthaison/pretokenized-retriever-v1-legal-question-answer-v1/pretokenized_retriever_v1.pt"
    # LoRA hyperparameters for PhoBERT (RoBERTa-style attention blocks).
    lora_r: int = 16
    lora_alpha: int = 32
    lora_dropout: float = 0.1
    lora_target_modules: List[str] = field(default_factory=lambda: ["query", "key", "value"])

    debug_checks: bool = False
    log_every: int = 20


config = TrainConfig()
if config.lora_r <= 0:
    raise ValueError("lora_r must be > 0")
if config.lora_alpha <= 0:
    raise ValueError("lora_alpha must be > 0")
if config.num_negatives <= 0:
    raise ValueError("num_negatives must be > 0")

print(config)
print(f"Effective train batch size = {config.batch_size * config.grad_accum_steps}")
print(
    f"LoRA setup | r={config.lora_r}, alpha={config.lora_alpha}, "
    f"dropout={config.lora_dropout}, targets={config.lora_target_modules}"
)
print(
    f"DataLoader workers={config.dataloader_num_workers}, "
    f"persistent_workers={config.dataloader_persistent_workers}"
)
print(f"Prepared negatives/query: {config.num_negatives}")
print(
    f"Candidate pool per query: all positives in batch + all prepared negatives in batch = "
    f"{config.batch_size} + ({config.batch_size} * {config.num_negatives}) = "
    f"{config.batch_size + (config.batch_size * config.num_negatives)}"
)
print(
    f"Negatives/query in this pool: (B-1) + (B*N) = {config.batch_size - 1} + "
    f"({config.batch_size} * {config.num_negatives}) = "
    f"{(config.batch_size - 1) + (config.batch_size * config.num_negatives)}"
)
print(f"Use pretokenized cache: {config.use_pretokenized_cache} ({config.pretokenized_cache_path})")

## Build Pairs and Split Data

In [ ]:
# ---------------------------------------------------------------------
# 2) Convert raw records to training pairs: (question, positive, negatives)
#    Current data schema: question, positive_chunks, negative_chunks
#    Here we enforce 1 question + 1 positive + num_negatives negatives.
# ---------------------------------------------------------------------
import inspect
from pathlib import Path
import importlib.util

repo_root = REPO_ROOT if "REPO_ROOT" in globals() else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

def _load_build_training_pairs():
    # Preferred import in local/Colab/Kaggle when repo root is available.
    try:
        from utils.retriever_data_utils import build_training_pairs as fn
        return fn
    except Exception:
        pass

    # Fallback: explicit path load (useful when running from attached Kaggle dataset).
    candidate_paths = [
        repo_root / "utils" / "retriever_data_utils.py",
        Path("/kaggle/input/datasets/trinhthaison/legal-question-answer-v1/utils/retriever_data_utils.py"),
    ]
    for mod_path in candidate_paths:
        if mod_path.exists():
            spec = importlib.util.spec_from_file_location("retriever_data_utils_dynamic", str(mod_path))
            if spec is not None and spec.loader is not None:
                module = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(module)
                return module.build_training_pairs

    return None


def _call_build_training_pairs_compat(fn, data, num_negatives):
    # Compatibility guard for helper signatures.
    params = inspect.signature(fn).parameters
    if "num_negatives" in params:
        return fn(data, num_negatives=num_negatives)

    pairs = fn(data)
    if not isinstance(pairs, list):
        raise TypeError("build_training_pairs must return a list of dict samples")

    negatives = []
    for row in pairs:
        negs = [n for n in row.get("negatives", []) if isinstance(n, str) and n.strip()]
        negatives.append(negs)
    return negatives 


def _build_pairs_manual(data, num_negatives):
    # Robust fallback for schema drifts across environments/files.
    pairs = []
    stats = {
        "total": 0,
        "missing_question": 0,
        "missing_positive": 0,
        "insufficient_negatives": 0,
    }

    for row in data:
        stats["total"] += 1

        q = row.get("question", "")
        if not isinstance(q, str) or not q.strip():
            stats["missing_question"] += 1
            continue

        pos = row.get("positive", None)
        if pos is None:
            stats["missing_positive"] += 1
            continue
        if not isinstance(pos, str):
            pos = str(pos)
        pos = pos.strip()

        neg_list = row.get("negatives", None)
        if neg_list is None:
            neg_list = row.get("negatives", [])
        if not isinstance(neg_list, list):
            neg_list = []
        neg_list = [n for n in neg_list if isinstance(n, str) and n.strip()]
        if len(neg_list) == 0:
            stats["insufficient_negatives"] += 1
            continue

        pairs.append(
            {
                "question": q.strip(),
                "positive": pos,
                "negatives": neg_list[:num_negatives],
            }
        )

    return pairs, stats


build_training_pairs = _load_build_training_pairs()
all_pairs = []

if build_training_pairs is not None:
    try:
        all_pairs = _call_build_training_pairs_compat(
            build_training_pairs,
            training_data,
            num_negatives=config.num_negatives,
        )
    except Exception as e:
        print(f"Helper build_training_pairs failed ({e}); falling back to manual builder.")

manual_pairs, manual_stats = _build_pairs_manual(training_data, config.num_negatives)
if not all_pairs:
    all_pairs = manual_pairs
    print("Using manual pair builder fallback.")
else:
    # If helper returned fewer pairs than manual builder, keep helper output but show warning.
    if len(all_pairs) < len(manual_pairs):
        print(
            f"Warning: helper built {len(all_pairs)} pairs, manual builder found {len(manual_pairs)} pairs."
        )

print(f"Total raw records: {len(training_data)}")
print(f"Total (question, positive) pairs: {len(all_pairs)}")
if all_pairs:
    print(f"Prepared negatives per pair: {len(all_pairs[0]['negatives'])}")
else:
    print(f"Pair-builder stats: {manual_stats}")
    raise ValueError(
        "No valid training pairs were built. Check question/positive_chunks/negative_chunks in your input data."
    )

if len(all_pairs) < 3:
    raise ValueError(
        f"Need at least 3 pairs to split train/val/test, but got {len(all_pairs)}."
    )

# --------------------------------------------------------
# 3) Split data train/val/test with ratio 70/15/15
# --------------------------------------------------------
train_pairs, temp_pairs = train_test_split(
    all_pairs,
    test_size=0.30,
    random_state=SEED,
    shuffle=True,
    stratify=None,
)

val_pairs, test_pairs = train_test_split(
    temp_pairs,
    test_size=0.50,
    random_state=SEED,
    shuffle=True,
    stratify=None,
)

print(f"Train size: {len(train_pairs)} ({len(train_pairs)/len(all_pairs):.2%})")
print(f"Val size:   {len(val_pairs)} ({len(val_pairs)/len(all_pairs):.2%})")
print(f"Test size:  {len(test_pairs)} ({len(test_pairs)/len(all_pairs):.2%})")

## Dataset and Tokenization Pipeline

In [ ]:
# --------------------------------------------------------------------
# 4) Dataset: prefer loading pretokenized cache, fallback to tokenize
# --------------------------------------------------------------------
import importlib.util

repo_root = REPO_ROOT if "REPO_ROOT" in globals() else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

def _load_module_from_candidates(module_name: str, candidate_files):
    for py_file in candidate_files:
        if py_file.exists():
            spec = importlib.util.spec_from_file_location(module_name, str(py_file))
            if spec is not None and spec.loader is not None:
                module = importlib.util.module_from_spec(spec)
                spec.loader.exec_module(module)
                return module
    return None

# Load pretokenized_dataset helpers robustly across local/Colab/Kaggle.
pretokenized_module = None
try:
    from utils import pretokenized_dataset as pretokenized_module
except Exception:
    pretokenized_module = _load_module_from_candidates(
        "pretokenized_dataset_dynamic",
        [
            repo_root / "utils" / "pretokenized_dataset.py",
            Path("/kaggle/input/datasets/trinhthaison/legal-question-answer-v1/utils/pretokenized_dataset.py"),
        ],
    )

if pretokenized_module is None:
    raise ModuleNotFoundError(
        "Cannot load pretokenized_dataset.py from utils. Ensure repo utils/ exists or Kaggle input includes it."
    )

PreTokenizedRetrieverDataset = pretokenized_module.PreTokenizedRetrieverDataset
load_pretokenized_cache = pretokenized_module.load_pretokenized_cache
validate_cache_meta = pretokenized_module.validate_cache_meta

# Load retriever_data_utils helpers robustly across local/Colab/Kaggle.
retriever_utils_module = None
try:
    from utils import retriever_data_utils as retriever_utils_module
except Exception:
    retriever_utils_module = _load_module_from_candidates(
        "retriever_data_utils_dynamic",
        [
            repo_root / "utils" / "retriever_data_utils.py",
            Path("/kaggle/input/datasets/trinhthaison/legal-question-answer-v1/utils/retriever_data_utils.py"),
        ],
    )

if retriever_utils_module is None:
    raise ModuleNotFoundError(
        "Cannot load retriever_data_utils.py from utils. Ensure repo utils/ exists or Kaggle input includes it."
    )

tokenize_multi_negative_pairs = retriever_utils_module.tokenize_multi_negative_pairs


class RetrieverPairDataset(Dataset):
    def __init__(
        self,
        pairs: List[Dict],
        tokenizer,
        max_q_len: int,
        max_c_len: int,
        num_negatives: int = 1,
    ):
        self.pairs = pairs
        self.num_negatives = num_negatives

        # Reuse shared tokenization logic from utils.
        tokenized = tokenize_multi_negative_pairs(
            pairs=self.pairs,
            tokenizer=tokenizer,
            max_q_len=max_q_len,
            max_c_len=max_c_len,
        )

        self.encoded_pairs = []
        for i, row in enumerate(self.pairs):
            self.encoded_pairs.append(
                {
                    "question": row["question"],
                    "positive": row["positive"],
                    "negatives": row["negatives"],
                    "q_tok": {
                        "input_ids": tokenized["q_input_ids"][i],
                        "attention_mask": tokenized["q_attention_mask"][i],
                    },
                    "p_tok": {
                        "input_ids": tokenized["p_input_ids"][i],
                        "attention_mask": tokenized["p_attention_mask"][i],
                    },
                    "n_tok": {
                        "input_ids": tokenized["n_input_ids"][i],
                        "attention_mask": tokenized["n_attention_mask"][i],
                    },
                }
            )

    def __len__(self):
        return len(self.encoded_pairs)

    def __getitem__(self, idx):
        return self.encoded_pairs[idx]


class CachedRetrieverPairDataset(Dataset):
    def __init__(self, split_cache: Dict[str, torch.Tensor], pairs: List[Dict]):
        self.tensor_ds = PreTokenizedRetrieverDataset(split_cache)
        self.pairs = pairs
        if len(self.tensor_ds) != len(self.pairs):
            raise ValueError(
                f"Cached split size ({len(self.tensor_ds)}) does not match pairs size ({len(self.pairs)})."
            )

    def __len__(self):
        return len(self.tensor_ds)

    def __getitem__(self, idx):
        item = self.tensor_ds[idx]
        row = self.pairs[idx]
        item["question"] = row["question"]
        item["positive"] = row["positive"]
        item["negatives"] = row["negatives"]

        # Backward compatibility: cached single-negative tensors [L] -> [1, L].
        if item["n_tok"]["input_ids"].dim() == 1:
            item["n_tok"]["input_ids"] = item["n_tok"]["input_ids"].unsqueeze(0)
            item["n_tok"]["attention_mask"] = item["n_tok"]["attention_mask"].unsqueeze(0)
        return item


def resolve_cache_path(preferred_path: str):
    base = Path(preferred_path)
    candidates = [
        Path("/kaggle/input/datasets/trinhthaison/pretokenized-retriever-v1-legal-question-answer-v1/pretokenized_retriever_v1.pt"),
        base,
        repo_root / preferred_path,
        Path.cwd() / preferred_path,
        (Path.cwd() / ".." / preferred_path).resolve(),
        Path("/content/Legal-question-answer-v1") / preferred_path,
        Path("/kaggle/working") / preferred_path,
        Path("/kaggle/input") / Path(preferred_path).name,
    ]

    for p in candidates:
        if p.exists():
            return p

    if IN_KAGGLE:
        found = list(Path("/kaggle/input").glob(f"**/{Path(preferred_path).name}"))
        if found:
            return found[0]

    return None


def build_datasets(
    train_pairs: List[Dict],
    val_pairs: List[Dict],
    test_pairs: List[Dict],
    tokenizer,
    cfg: TrainConfig,
    seed: int,
):
    if cfg.use_pretokenized_cache:
        cache_path = resolve_cache_path(cfg.pretokenized_cache_path)
        if cache_path is not None:
            try:
                loaded = load_pretokenized_cache(str(cache_path), map_location="cpu")
                validate_cache_meta(
                    loaded,
                    model_name=cfg.model_name,
                    max_q_len=cfg.max_q_len,
                    max_c_len=cfg.max_c_len,
                )
                cached_seed = loaded.get("meta", {}).get("seed", None)
                if cached_seed is not None and cached_seed != seed:
                    raise ValueError(f"seed mismatch: expected {seed}, got {cached_seed}")

                n_shape = loaded["train"]["n_input_ids"].shape
                # Require cache to match current multi-negative setup.
                if len(n_shape) != 3 or n_shape[1] != cfg.num_negatives:
                    raise ValueError(
                        f"Cache n_input_ids shape {tuple(n_shape)} mismatches num_negatives={cfg.num_negatives}"
                    )

                train_ds = CachedRetrieverPairDataset(loaded["train"], train_pairs)
                val_ds = CachedRetrieverPairDataset(loaded["val"], val_pairs)
                test_ds = CachedRetrieverPairDataset(loaded["test"], test_pairs)
                print(f"Using pretokenized cache from: {cache_path}")
                return train_ds, val_ds, test_ds
            except Exception as e:
                print(f"Pretokenized cache unavailable or invalid ({e}); fallback to on-the-fly tokenization.")
        else:
            print("Pretokenized cache file not found; fallback to on-the-fly tokenization.")

    train_ds = RetrieverPairDataset(
        train_pairs,
        tokenizer=tokenizer,
        max_q_len=cfg.max_q_len,
        max_c_len=cfg.max_c_len,
        num_negatives=cfg.num_negatives,
    )
    val_ds = RetrieverPairDataset(
        val_pairs,
        tokenizer=tokenizer,
        max_q_len=cfg.max_q_len,
        max_c_len=cfg.max_c_len,
        num_negatives=cfg.num_negatives,
    )
    test_ds = RetrieverPairDataset(
        test_pairs,
        tokenizer=tokenizer,
        max_q_len=cfg.max_q_len,
        max_c_len=cfg.max_c_len,
        num_negatives=cfg.num_negatives,
    )
    return train_ds, val_ds, test_ds


tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)

# RoBERTa/PhoBERT positional embedding size is typically 258, meaning safe max tokenized length is 256.
model_max_len = getattr(tokenizer, "model_max_length", None)
if isinstance(model_max_len, int) and model_max_len > 0 and model_max_len < 10_000:
    if config.max_q_len > model_max_len or config.max_c_len > model_max_len:
        raise ValueError(
            f"max_q_len/max_c_len must be <= tokenizer.model_max_length ({model_max_len}). "
            f"Got max_q_len={config.max_q_len}, max_c_len={config.max_c_len}."
        )
    print(f"Tokenizer max length: {model_max_len} | q_len={config.max_q_len}, c_len={config.max_c_len}")
else:
    # Conservative fallback for PhoBERT-like models if tokenizer does not expose a practical value.
    if config.max_q_len > 256 or config.max_c_len > 256:
        raise ValueError(
            f"For PhoBERT, set max_q_len/max_c_len <= 256. "
            f"Got max_q_len={config.max_q_len}, max_c_len={config.max_c_len}."
        )


def collate_fn(batch: List[Dict]):
    q_tok = {
        "input_ids": torch.stack([x["q_tok"]["input_ids"] for x in batch], dim=0),
        "attention_mask": torch.stack([x["q_tok"]["attention_mask"] for x in batch], dim=0),
    }
    p_tok = {
        "input_ids": torch.stack([x["p_tok"]["input_ids"] for x in batch], dim=0),
        "attention_mask": torch.stack([x["p_tok"]["attention_mask"] for x in batch], dim=0),
    }
    n_tok = {
        "input_ids": torch.stack([x["n_tok"]["input_ids"] for x in batch], dim=0),
        "attention_mask": torch.stack([x["n_tok"]["attention_mask"] for x in batch], dim=0),
    }

    return {
        "q_tok": q_tok,
        "p_tok": p_tok,
        "n_tok": n_tok,
    }

## Dual Encoder and Epoch Runner

In [ ]:
import time
from contextlib import nullcontext
from tqdm.auto import tqdm


class DualEncoder(nn.Module):
    def __init__(self, model_name: str):
        super().__init__()
        # Dual-encoder: one encoder for questions, one encoder for chunks.
        self.q_encoder = AutoModel.from_pretrained(model_name)
        self.d_encoder = AutoModel.from_pretrained(model_name)

    @staticmethod
    def mean_pool(last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = torch.sum(last_hidden_state * mask, dim=1)
        denom = torch.clamp(mask.sum(dim=1), min=1e-9)
        return summed / denom

    @staticmethod
    def _validate_token_ids(tok, vocab_size: int, name: str):
        input_ids = tok["input_ids"]
        min_id = int(input_ids.min().item())
        max_id = int(input_ids.max().item())
        if min_id < 0 or max_id >= vocab_size:
            raise RuntimeError(
                f"{name} has out-of-range token ids: min={min_id}, max={max_id}, vocab_size={vocab_size}"
            )

    def encode_questions(self, q_tok):
        out = self.q_encoder(
            input_ids=q_tok["input_ids"],
            attention_mask=q_tok["attention_mask"],
        )
        emb = self.mean_pool(out.last_hidden_state, q_tok["attention_mask"])
        return F.normalize(emb, p=2, dim=-1)

    def encode_docs(self, d_tok):
        out = self.d_encoder(
            input_ids=d_tok["input_ids"],
            attention_mask=d_tok["attention_mask"],
        )
        emb = self.mean_pool(out.last_hidden_state, d_tok["attention_mask"])
        return F.normalize(emb, p=2, dim=-1)

    def forward(self, batch):
        q_tok = {k: v.to(device) for k, v in batch["q_tok"].items()}
        p_tok = {k: v.to(device) for k, v in batch["p_tok"].items()}
        n_tok = {k: v.to(device) for k, v in batch["n_tok"].items()}

        # Backward compatibility in case n_tok arrives as [B, L].
        if n_tok["input_ids"].dim() == 2:
            n_tok["input_ids"] = n_tok["input_ids"].unsqueeze(1)
            n_tok["attention_mask"] = n_tok["attention_mask"].unsqueeze(1)

        if config.debug_checks:
            q_vocab_size = self.q_encoder.config.vocab_size
            d_vocab_size = self.d_encoder.config.vocab_size
            self._validate_token_ids(q_tok, q_vocab_size, "Question tokens")
            self._validate_token_ids(p_tok, d_vocab_size, "Positive doc tokens")
            self._validate_token_ids(
                {
                    "input_ids": n_tok["input_ids"].reshape(-1, n_tok["input_ids"].size(-1)),
                    "attention_mask": n_tok["attention_mask"].reshape(-1, n_tok["attention_mask"].size(-1)),
                },
                d_vocab_size,
                "Negative doc tokens",
            )

        q_emb = self.encode_questions(q_tok)        # [B, H]
        p_emb = self.encode_docs(p_tok)             # [B, H]

        bsz, num_neg, seq_len = n_tok["input_ids"].shape
        n_tok_flat = {
            "input_ids": n_tok["input_ids"].reshape(bsz * num_neg, seq_len),
            "attention_mask": n_tok["attention_mask"].reshape(bsz * num_neg, seq_len),
        }
        n_emb = self.encode_docs(n_tok_flat).reshape(bsz, num_neg, -1)  # [B, N, H]

        # Global pool per query: all positives in batch + all prepared negatives across queries.
        n_pool = n_emb.reshape(bsz * num_neg, -1)  # [B*N, H]
        candidate_emb = torch.cat([p_emb, n_pool], dim=0)  # [B + B*N, H]
        logits = torch.matmul(q_emb, candidate_emb.transpose(0, 1)) / config.temperature  # [B, B + B*N]
        labels = torch.arange(bsz, device=device, dtype=torch.long)
        # Correct positive for query i is column i in the first B columns.


        if config.debug_checks and not torch.isfinite(logits).all():
            raise RuntimeError("Non-finite logits detected (NaN/Inf).")

        loss = F.cross_entropy(logits, labels)
        if config.debug_checks and not torch.isfinite(loss):
            raise RuntimeError("Non-finite loss detected (NaN/Inf).")

        preds = torch.argmax(logits, dim=1)
        acc = (preds == labels).float().mean().item()
        return loss, acc


def run_epoch(
    model,
    loader,
    optimizer=None,
    scheduler=None,
    train_mode=True,
    scaler=None,
    grad_accum_steps: int = 1,
    use_amp: bool = True,
    amp_dtype: torch.dtype = torch.float16,
    epoch_idx: int = 0,
    show_progress: bool = True,
    log_every: int = 20,
):
    if train_mode:
        model.train()
        if optimizer is None:
            raise ValueError("optimizer is required when train_mode=True")
        optimizer.zero_grad(set_to_none=True)
    else:
        model.eval()

    total_loss = 0.0
    total_acc = 0.0
    steps = 0

    autocast_enabled = bool(use_amp and torch.cuda.is_available())
    phase = "train" if train_mode else "val/test"
    pbar = tqdm(
        enumerate(loader),
        total=len(loader),
        desc=f"Epoch {epoch_idx:02d} [{phase}]",
        leave=False,
        disable=not show_progress,
    )

    for step_idx, batch in pbar:
        try:
            with torch.set_grad_enabled(train_mode):
                autocast_ctx = (
                    torch.autocast(device_type="cuda", dtype=amp_dtype, enabled=True)
                    if autocast_enabled
                    else nullcontext()
                )
                with autocast_ctx:
                    loss, acc = model(batch)

                if train_mode:
                    loss_to_backprop = loss / max(grad_accum_steps, 1)

                    if scaler is not None and autocast_enabled:
                        scaler.scale(loss_to_backprop).backward()
                    else:
                        loss_to_backprop.backward()

                    should_step = (
                        ((step_idx + 1) % max(grad_accum_steps, 1) == 0)
                        or ((step_idx + 1) == len(loader))
                    )

                    if should_step:
                        if scaler is not None and autocast_enabled:
                            old_scale = scaler.get_scale()
                            scaler.unscale_(optimizer)
                            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
                            scaler.step(optimizer)
                            scaler.update()
                            optimizer_was_run = scaler.get_scale() >= old_scale
                        else:
                            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
                            optimizer.step()
                            optimizer_was_run = True

                        optimizer.zero_grad(set_to_none=True)
                        if scheduler is not None and optimizer_was_run:
                            scheduler.step()

            total_loss += loss.item()
            total_acc += acc
            steps += 1

            if steps % max(log_every, 1) == 0 or steps == len(loader):
                avg_loss = total_loss / max(steps, 1)
                avg_acc = total_acc / max(steps, 1)
                pbar.set_postfix(loss=f"{avg_loss:.4f}", acc=f"{avg_acc:.4f}")

        except RuntimeError as e:
            msg = str(e)
            if "device-side assert" in msg or "out-of-range token ids" in msg or "Non-finite" in msg:
                print(f"Runtime error at step {step_idx}: {msg}")
                print("Hint: inspect raw batch texts and token ranges for this step.")
            raise

    return total_loss / max(steps, 1), total_acc / max(steps, 1)

## Build Dataloaders and Initialize LoRA Model

In [ ]:
# -------------------------
# 5) Build dataloaders
# -------------------------
train_ds, val_ds, test_ds = build_datasets(
    train_pairs=train_pairs,
    val_pairs=val_pairs,
    test_pairs=test_pairs,
    tokenizer=tokenizer,
    cfg=config,
    seed=SEED,
 )

num_workers = max(4, int(config.dataloader_num_workers))
use_persistent_workers = bool(config.dataloader_persistent_workers and num_workers > 0)

train_loader = DataLoader(
    train_ds,
    batch_size=config.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
    num_workers=num_workers,
    persistent_workers=use_persistent_workers,
 )
val_loader = DataLoader(
    val_ds,
    batch_size=config.batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
    num_workers=num_workers,
    persistent_workers=use_persistent_workers,
 )
test_loader = DataLoader(
    test_ds,
    batch_size=config.batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
    num_workers=num_workers,
    persistent_workers=use_persistent_workers,
 )

# -------------------------------
# 6) Model + LoRA + optimizers
# -------------------------------
model = DualEncoder(config.model_name).to(device)

if hasattr(model.q_encoder, "gradient_checkpointing_enable"):
    model.q_encoder.gradient_checkpointing_enable()
if hasattr(model.d_encoder, "gradient_checkpointing_enable"):
    model.d_encoder.gradient_checkpointing_enable()

# Transformers warning fix: use_cache must be disabled with gradient checkpointing.
if hasattr(model.q_encoder, "config") and hasattr(model.q_encoder.config, "use_cache"):
    model.q_encoder.config.use_cache = False
if hasattr(model.d_encoder, "config") and hasattr(model.d_encoder.config, "use_cache"):
    model.d_encoder.config.use_cache = False


def apply_lora_to_encoder(hf_model, cfg: TrainConfig):
    lora_cfg = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        r=cfg.lora_r,
        lora_alpha=cfg.lora_alpha,
        lora_dropout=cfg.lora_dropout,
        target_modules=cfg.lora_target_modules,
        bias="none",
    )
    return get_peft_model(hf_model, lora_cfg)


model.q_encoder = apply_lora_to_encoder(model.q_encoder, config)
model.d_encoder = apply_lora_to_encoder(model.d_encoder, config)

# Keep only LoRA adapter params trainable (PEFT handles this, but assert for safety).
trainable_params = [p for p in model.parameters() if p.requires_grad]
if not trainable_params:
    raise RuntimeError("No trainable parameters found after applying LoRA.")

optimizer = torch.optim.AdamW(
    trainable_params,
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
 )

steps_per_epoch = math.ceil(len(train_loader) / config.grad_accum_steps)
total_training_steps = steps_per_epoch * config.max_epochs
warmup_steps = int(total_training_steps * config.warmup_ratio)
linear_scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
 )

plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=1,
    min_lr=1e-7,
 )

# AMP policy: prefer BF16 on supported GPUs; fallback to FP16 + GradScaler on older CUDA.
if torch.cuda.is_available():
    if torch.cuda.is_bf16_supported():
        use_amp = True
        amp_dtype = torch.bfloat16
        scaler = None  # BF16 does not need gradient scaling.
        amp_mode = "bf16"
    else:
        use_amp = True
        amp_dtype = torch.float16
        scaler = torch.amp.GradScaler("cuda", enabled=True)
        amp_mode = "fp16"
else:
    use_amp = False
    amp_dtype = torch.float32
    scaler = None
    amp_mode = "off (cpu)"

total_params = sum(p.numel() for p in model.parameters())
trainable_param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
trainable_ratio = 100.0 * trainable_param_count / max(total_params, 1)

print(f"Total optimizer update steps: {total_training_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"AMP mode: {amp_mode}")
print(f"Gradient accumulation steps: {config.grad_accum_steps}")
print(f"LoRA target modules: {config.lora_target_modules}")
print(f"Trainable params: {trainable_param_count:,} / {total_params:,} ({trainable_ratio:.2f}%)")
print(
    f"use_cache flags | q_encoder={getattr(model.q_encoder.config, 'use_cache', None)} | "
    f"d_encoder={getattr(model.d_encoder.config, 'use_cache', None)}"
)

# Optional sanity summary from PEFT wrappers.
try:
    model.q_encoder.print_trainable_parameters()
    model.d_encoder.print_trainable_parameters()
except Exception:
    pass

## Train, Validate, and Test

In [ ]:
# -------------------------------
# 7) Train + validation + test
# -------------------------------
import copy
import time

best_val_loss = float("inf")
best_state = None
best_ckpt = None
best_epoch = None
epochs_no_improve = 0
history = []

for epoch in range(1, config.max_epochs + 1):
    epoch_start = time.time()

    train_loss, train_acc = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scheduler=linear_scheduler,
        train_mode=True,
        scaler=scaler,
        grad_accum_steps=config.grad_accum_steps,
        use_amp=use_amp,
        amp_dtype=amp_dtype,
        epoch_idx=epoch,
        show_progress=True,
        log_every=config.log_every,
    )

    with torch.no_grad():
        val_loss, val_acc = run_epoch(
            model,
            val_loader,
            train_mode=False,
            use_amp=use_amp,
            amp_dtype=amp_dtype,
            epoch_idx=epoch,
            show_progress=True,
            log_every=config.log_every,
        )

    plateau_scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]["lr"]
    epoch_sec = time.time() - epoch_start

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "lr": current_lr,
            "epoch_seconds": epoch_sec,
        }
    )

    print(
        f"Epoch {epoch:02d} | "
        f"train_loss={train_loss:.4f} train_acc={train_acc:.4f} | "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} | "
        f"lr={current_lr:.2e} | time={epoch_sec/60:.2f} min"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())
        best_ckpt = {
            "epoch": epoch,
            "best_val_loss": best_val_loss,
            "model_state_dict": copy.deepcopy(model.state_dict()),
            "optimizer_state_dict": copy.deepcopy(optimizer.state_dict()),
            "linear_scheduler_state_dict": copy.deepcopy(linear_scheduler.state_dict()),
            "plateau_scheduler_state_dict": copy.deepcopy(plateau_scheduler.state_dict()),
            "scaler_state_dict": copy.deepcopy(scaler.state_dict()) if scaler is not None else None,
            "config": vars(config),
            "seed": SEED,
            "model_name": config.model_name,
            "history": copy.deepcopy(history),
        }
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= config.early_stopping_patience:
            print(f"Early stopping triggered at epoch {epoch}.")
            break

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

if best_state is not None:
    model.load_state_dict(best_state)

with torch.no_grad():
    test_loss, test_acc = run_epoch(
        model,
        test_loader,
        train_mode=False,
        use_amp=use_amp,
        amp_dtype=amp_dtype,
        epoch_idx=epoch,
        show_progress=True,
        log_every=config.log_every,
    )

print("\nFinal evaluation on test set:")
print(f"Best epoch: {best_epoch} | Best val loss: {best_val_loss:.4f}")
print(f"Test loss: {test_loss:.4f}")
print(f"Test acc:  {test_acc:.4f}")

In [ ]:
import matplotlib.pyplot as plt

if "history" not in globals() or not history:
    print("Run the training cell first so 'history' is available.")
else:
    epochs = [row["epoch"] for row in history]
    train_loss_curve = [row["train_loss"] for row in history]
    val_loss_curve = [row["val_loss"] for row in history]

    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_loss_curve, marker="o", linewidth=2, label="Train loss")
    plt.plot(epochs, val_loss_curve, marker="s", linewidth=2, label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Retriever Training Loss Curve")
    plt.xticks(epochs)
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

## Train Loop and Final Evaluation

## Save Best Resume Checkpoint

In [ ]:
# Save only the best checkpoint for future hard-negative retraining
import os
import sys
from pathlib import Path

import torch

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle").exists()

if IN_COLAB:
    # Persist artifacts on Google Drive if mounted.
    save_root = "/content/drive/MyDrive/artifacts_dual_encoder_checkpoint"
elif IN_KAGGLE:
    # Kaggle only allows writing under /kaggle/working.
    save_root = "/kaggle/working/artifacts_dual_encoder_checkpoint"
else:
    save_root = "artifacts_dual_encoder_checkpoint"

checkpoint_path = os.path.join(save_root, "best_dual_encoder_resume_v1.pt")

required = ["model", "optimizer", "linear_scheduler", "plateau_scheduler", "config", "SEED"]
missing = [name for name in required if name not in globals()]
if missing:
    print(f"Run the training setup first. Missing objects: {missing}")
else:
    os.makedirs(save_root, exist_ok=True)

    if "best_ckpt" in globals() and best_ckpt is not None:
        ckpt_payload = best_ckpt
    else:
        # Fallback: save current state if best checkpoint dict is unavailable.
        ckpt_payload = {
            "epoch": None,
            "best_val_loss": None,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "linear_scheduler_state_dict": linear_scheduler.state_dict(),
            "plateau_scheduler_state_dict": plateau_scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict() if "scaler" in globals() and scaler is not None else None,
            "config": vars(config),
            "seed": SEED,
            "model_name": config.model_name,
            "history": history if "history" in globals() else [],
        }

    torch.save(ckpt_payload, checkpoint_path)

    print("Saved best resume checkpoint:")
    print(f"- {checkpoint_path}")
    print("Includes: model_state_dict, optimizer, schedulers, scaler, config, seed, history")
    print("Use this file to continue training on newly mined hard negatives.")

    if IN_COLAB:
        try:
            from google.colab import files  # type: ignore
            files.download(checkpoint_path)
        except Exception:
            print("Colab auto-download unavailable. Use the printed checkpoint path.")
    elif IN_KAGGLE:
        print("Kaggle: download from the right panel -> Output -> /kaggle/working.")

In [ ]:
# Test-set retrieval metrics: Recall@k, MAP@k, NDCG@k
import numpy as np
import torch


def evaluate_retriever_test_metrics(model, test_ds, tokenizer, device, config, k_values=(1, 3, 5)):
    model.eval()
    k_values = sorted(set(int(k) for k in k_values if int(k) > 0))

    total_samples = 0
    recall_hits = {k: 0 for k in k_values}
    ap_sums = {k: 0.0 for k in k_values}
    ndcg_sums = {k: 0.0 for k in k_values}

    with torch.no_grad():
        for idx in range(len(test_ds)):
            sample = test_ds[idx]
            candidates = [sample["positive"]] + sample["negatives"]
            if len(candidates) <= 1:
                continue

            q_tok = tokenizer(
                [sample["question"]],
                padding=True,
                truncation=True,
                max_length=config.max_q_len,
                return_tensors="pt",
            )
            c_tok = tokenizer(
                candidates,
                padding=True,
                truncation=True,
                max_length=config.max_c_len,
                return_tensors="pt",
            )

            q_tok = {k: v.to(device) for k, v in q_tok.items()}
            c_tok = {k: v.to(device) for k, v in c_tok.items()}

            q_emb = model.encode_questions(q_tok)
            c_emb = model.encode_docs(c_tok)

            logits = torch.matmul(q_emb, c_emb.transpose(0, 1)).squeeze(0) / config.temperature
            scores = logits.detach().cpu().numpy()

            # Positive doc is always index 0.
            rank_order = np.argsort(-scores)
            pos_rank = int(np.where(rank_order == 0)[0][0]) + 1

            total_samples += 1

            for k in k_values:
                k_eff = min(k, len(candidates))
                hit = int(pos_rank <= k_eff)

                # Single-positive setting:
                # - Recall@k = 1 if positive in top-k else 0
                # - AP@k = 1/rank if positive in top-k else 0
                # - NDCG@k = 1/log2(rank+1) if positive in top-k else 0
                recall_hits[k] += hit
                ap_sums[k] += (1.0 / pos_rank) if hit else 0.0
                ndcg_sums[k] += (1.0 / np.log2(pos_rank + 1.0)) if hit else 0.0

    if total_samples == 0:
        return {
            "evaluated_samples": 0,
            "note": "No test samples with at least 1 negative candidate.",
        }

    metrics = {"evaluated_samples": total_samples}
    for k in k_values:
        metrics[f"recall@{k}"] = float(recall_hits[k] / total_samples)
        metrics[f"map@{k}"] = float(ap_sums[k] / total_samples)
        metrics[f"ndcg@{k}"] = float(ndcg_sums[k] / total_samples)

    return metrics


if "model" not in globals() or "test_ds" not in globals():
    print("Run training/evaluation cells first so model and test_ds are available.")
else:
    test_metrics = evaluate_retriever_test_metrics(
        model=model,
        test_ds=test_ds,
        tokenizer=tokenizer,
        device=device,
        config=config,
        k_values=(1, 3, 5),
    )

    print("Test set metrics (Recall/MAP/NDCG):")
    for key, value in test_metrics.items():
        if isinstance(value, float):
            print(f"- {key}: {value:.4f}")
        else:
            print(f"- {key}: {value}")

    test_metrics

## Error Analysis on Test Set

In [ ]:
# # Error analysis: inspect wrong samples on test set
# def _short_text(text: str, max_len: int = 220) -> str:
#     text = " ".join(text.split())
#     if len(text) <= max_len:
#         return text
#     return text[:max_len] + "..."


# def collect_wrong_test_samples(model, test_ds, tokenizer, device, config):
#     model.eval()
#     wrong_cases = []

#     with torch.no_grad():
#         for idx in range(len(test_ds)):
#             sample = test_ds[idx]
#             candidates = [sample["positive"]] + sample["negatives"]

#             if len(candidates) == 1:
#                 # No negatives for this sample -> cannot become a wrong ranking case.
#                 continue

#             q_tok = tokenizer(
#                 [sample["question"]],
#                 padding=True,
#                 truncation=True,
#                 max_length=config.max_q_len,
#                 return_tensors="pt",
#             )
#             c_tok = tokenizer(
#                 candidates,
#                 padding=True,
#                 truncation=True,
#                 max_length=config.max_c_len,
#                 return_tensors="pt",
#             )

#             q_tok = {k: v.to(device) for k, v in q_tok.items()}
#             c_tok = {k: v.to(device) for k, v in c_tok.items()}

#             q_emb = model.encode_questions(q_tok)  # [1, H]
#             c_emb = model.encode_docs(c_tok)       # [num_candidates, H]

#             logits = torch.matmul(q_emb, c_emb.transpose(0, 1)).squeeze(0) / config.temperature
#             probs = torch.softmax(logits, dim=0)

#             pred_idx = int(torch.argmax(logits).item())
#             if pred_idx != 0:
#                 wrong_cases.append(
#                     {
#                         "test_index": idx,
#                         "question": sample["question"],
#                         "positive_chunk": sample["positive"],
#                         "predicted_index": pred_idx,
#                         "predicted_chunk": candidates[pred_idx],
#                         "positive_score": float(logits[0].item()),
#                         "predicted_score": float(logits[pred_idx].item()),
#                         "score_gap": float((logits[pred_idx] - logits[0]).item()),
#                         "positive_prob": float(probs[0].item()),
#                         "predicted_prob": float(probs[pred_idx].item()),
#                         "num_negatives": len(sample["negatives"]),
#                     }
#                 )

#     return wrong_cases


# if "model" not in globals() or "test_ds" not in globals():
#     print("Run the training + test evaluation cells first, then run this cell.")
# else:
#     wrong_cases = collect_wrong_test_samples(model, test_ds, tokenizer, device, config)
#     total_eval = len([test_ds[i] for i in range(len(test_ds)) if len(test_ds[i]["negatives"]) > 0])
#     wrong_count = len(wrong_cases)
#     wrong_rate = (wrong_count / total_eval * 100) if total_eval > 0 else 0.0

#     print(f"Evaluated test samples (with >=1 negative): {total_eval}")
#     print(f"Wrong ranking cases: {wrong_count} ({wrong_rate:.2f}%)")

#     preview_n = 10
#     for i, case in enumerate(wrong_cases[:preview_n], start=1):
#         print("\n" + "=" * 90)
#         print(f"Case {i} | test_index={case['test_index']} | num_negatives={case['num_negatives']}")
#         print(f"Score gap (pred - pos): {case['score_gap']:.4f}")
#         print(f"Positive prob: {case['positive_prob']:.4f} | Pred prob: {case['predicted_prob']:.4f}")
#         print("Question:", _short_text(case["question"]))
#         print("Positive:", _short_text(case["positive_chunk"]))
#         print("Predicted:", _short_text(case["predicted_chunk"]))

#     wrong_cases